
# Gunshot Similarity Search (CNN embeddings) — Kaggle gunshots → scan lab WAVs (5s clips)

This notebook fixes the `wget: command not found` issue by using **Python `urllib`** to fetch the small metadata
and **PANNs Cnn14** checkpoint that `panns_inference` expects.

If your environment has no internet access, you can manually download these two files and place them in:
`~/panns_data/`
- `class_labels_indices.csv`
- `Cnn14_mAP=0.431.pth`

Model: **PANNs Cnn14 (CNN)** embeddings + **cosine similarity** to a bank of gunshot embeddings built from your Kaggle dataset.


In [1]:

# ========================
# 0) INSTALL (run once)
# ========================
!pip install -U torch torchaudio panns-inference librosa soundfile numpy pandas matplotlib tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 22.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 736.4/736.4 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.3 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: pandas90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [torch]
    Found existing installation: pandas 2.2.3━━━━━━━━━━━━━━━━━ 1/6 [torch]
    Uninstalling pandas-2.2.3:╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/6 [pandas]
      Successfully uninstalled pandas-2.2.3━━━━━━━━━━━━━━━━━━━ 2/6 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [panns-inference] [torchaudio]


In [13]:

# ========================
# 1) SETTINGS (EDIT ME)
# ========================

from pathlib import Path

KAGGLE_ROOT = Path.home() / "Desktop" / "archive"
LAB_ROOT = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53/02-11-25 to 02-25-25")

WINDOW_SEC = 5.0
HOP_SEC = 2.5
TOP_K_PER_FILE = 5
SIM_THRESHOLD = 0.65

USE_LOCAL_CACHE = True
CACHE_DIR = Path("wav_cache_temp")
CACHE_DIR.mkdir(exist_ok=True)

OUT_DIR = Path("outputs")
CLIP_DIR = OUT_DIR / "clips"
SPEC_DIR = OUT_DIR / "specs"
CLIP_DIR.mkdir(parents=True, exist_ok=True)
SPEC_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE_ROOT:", KAGGLE_ROOT)
print("LAB_ROOT:", LAB_ROOT)


KAGGLE_ROOT: /Users/minyoungkim/Desktop/archive
LAB_ROOT: /Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53/02-11-25 to 02-25-25


In [3]:

# ========================
# 2) PREP: make PANNs files available WITHOUT wget
# ========================

import urllib.request
from pathlib import Path

PANN_DIR = Path.home() / "panns_data"
PANN_DIR.mkdir(exist_ok=True)

LABELS_URL = "https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"
LABELS_PATH = PANN_DIR / "class_labels_indices.csv"

if not LABELS_PATH.exists():
    print("Downloading:", LABELS_URL)
    urllib.request.urlretrieve(LABELS_URL, LABELS_PATH)
else:
    print("Found:", LABELS_PATH)

CKPT_URL = "https://zenodo.org/record/3987831/files/Cnn14_mAP=0.431.pth?download=1"
CKPT_PATH = PANN_DIR / "Cnn14_mAP=0.431.pth"

if not CKPT_PATH.exists():
    print("Downloading:", CKPT_URL)
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
else:
    print("Found:", CKPT_PATH)

print("PANN_DIR ready:", PANN_DIR)


Found: /Users/minyoungkim/panns_data/class_labels_indices.csv
Found: /Users/minyoungkim/panns_data/Cnn14_mAP=0.431.pth
PANN_DIR ready: /Users/minyoungkim/panns_data


In [5]:

# ========================
# 3) IMPORTS + MODEL LOAD
# ========================

import time
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm

from panns_inference import AudioTagging

# Use the checkpoint we downloaded (avoids any auto-download logic)
at = AudioTagging(checkpoint_path=str(CKPT_PATH), device='cpu')

PANN_SR = 32000


Checkpoint path: /Users/minyoungkim/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [14]:

# ========================
# 4) FILE DISCOVERY
# ========================

from pathlib import Path

def list_wavs(root: Path):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Path does not exist: {root}")
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".wav"])

gunshot_wavs = list_wavs(KAGGLE_ROOT)
lab_wavs = list_wavs(LAB_ROOT)

print("Gunshot WAVs found:", len(gunshot_wavs))
print("Lab WAVs found:", len(lab_wavs))
print("Example gunshot:", gunshot_wavs[:2])
print("Example lab:", lab_wavs[:2])


Gunshot WAVs found: 851
Lab WAVs found: 84
Example gunshot: [PosixPath('/Users/minyoungkim/Desktop/archive/AK-12/3 (1).wav'), PosixPath('/Users/minyoungkim/Desktop/archive/AK-12/3 (10).wav')]
Example lab: [PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53/02-11-25 to 02-25-25/20250211/20250211_180000.WAV'), PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53/02-11-25 to 02-25-25/20250211/20250211_190000.WAV')]


In [15]:
import os
print("cwd:", os.getcwd())

cwd: /Users/minyoungkim/Desktop/elephant_acoustic_monitoring


In [8]:
import os
print(os.listdir("/Volumes"))

['Macintosh HD', 'aid_elephants_interaction']


In [9]:
from pathlib import Path

root = Path("/Volumes/aid_elephants_interaction")
print(root.exists())
print([p.name for p in root.iterdir()][:50])

True
['#recycle', 'Documents Folder', 'Audio Data', 'Audio_Overview.pptx', '.DS_Store', 'elephant_rumble_results', 'yamnet_results', 'elephant_photos']


In [10]:
p = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53")
print(p.exists())
print([x.name for x in p.iterdir() if x.is_dir()])

True
['02-11-25 to 02-25-25', '02-25-25 to 03-11-25', '03_11_25 to 03_25_25', '04-22-25 to 05-06-25', '03-25-25 to 04-22-25', '05_06_25 to _05_19_25']


In [16]:

# ========================
# 5) TEMP CACHE HELPERS
# ========================

from pathlib import Path

def cached_path_for(src: Path, cache_dir: Path) -> Path:
    return cache_dir / src.name

def ensure_local_copy(src: Path, cache_dir: Path, retries=3, sleep_sec=6) -> Path | None:
    src = Path(src)
    dst = cached_path_for(src, cache_dir)

    try:
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            return dst
    except Exception:
        pass

    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(src, dst)
            return dst
        except Exception as e:
            print(f"[cache] copy failed {attempt}/{retries} for {src.name}: {type(e).__name__}: {e}")
            time.sleep(sleep_sec)

    print(f"[cache] SKIPPING (unreadable right now): {src}")
    return None

def safe_unlink(p: Path):
    try:
        if p and Path(p).exists():
            Path(p).unlink()
    except Exception as e:
        print(f"[cache] couldn't delete {p}: {e}")


In [ ]:

# ========================
# 6) AUDIO HELPERS + EMBEDDINGS
# ========================

def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    return np.mean(x, axis=1)

def load_segment(path, start_sec: float, dur_sec: float):
    """Try soundfile first; if it fails, fall back to librosa."""
    path = str(path)

    # 1) soundfile path (fast)
    try:
        with sf.SoundFile(path) as f:
            sr = f.samplerate
            start_frame = int(start_sec * sr)
            n_frames = int(dur_sec * sr)
            start_frame = max(0, min(start_frame, len(f) - 1))
            f.seek(start_frame)
            data = f.read(n_frames, dtype="float32", always_2d=True)
            y = data.mean(axis=1)  # mono
            return y, sr
    except Exception:
        pass

    y, sr = librosa.load(path, sr=None, mono=True, offset=start_sec, duration=dur_sec)
    return y.astype(np.float32), sr

def resample(y: np.ndarray, orig_sr: int, target_sr: int):
    if orig_sr == target_sr:
        return y.astype(np.float32)
    return librosa.resample(y.astype(np.float32), orig_sr=orig_sr, target_sr=target_sr)


def embed_panns(y: np.ndarray, sr: int):
    y32 = resample(y, sr, PANN_SR)

    # too-short audio -> skip
    if len(y32) < int(0.25 * PANN_SR):
        return None

    y32 = np.asarray(y32, dtype=np.float32)

    # IMPORTANT: add batch dimension
    x = y32[None, :]   # shape (1, n_samples)

    clipwise, embedding = at.inference(x)

    # embedding could be torch tensor or numpy
    if hasattr(embedding, "detach"):
        embedding = embedding.detach().cpu().numpy()

    embedding = np.asarray(embedding)  # expect (1, 2048)
    e = embedding[0].astype(np.float32)

    e /= (np.linalg.norm(e) + 1e-9)
    return e



In [ ]:

# ========================
# 7) BUILD GUNSHOT EMBEDDING BANK
# ========================

def build_gunshot_bank(paths, max_files=None):
    bank = []
    used = paths if max_files is None else paths[:max_files]
    for p in tqdm(used, desc="Embedding gunshot dataset"):
        try:
            p = gunshot_wavs[0]
            y, sr = load_segment(p, 0.0, WINDOW_SEC)
            print("Loaded:", p, "sr=", sr, "len=", len(y))
            e = embed_panns(y, sr)
            print("Embedding:", None if e is None else e.shape)
            if e is None:
                continue
            bank.append(e)
        except Exception as ex:
            print(f"[gunshot] skip {p.name}: {type(ex).__name__}: {ex}")
    if len(bank) == 0:
        raise RuntimeError("No gunshot embeddings created. Check KAGGLE_ROOT.")
    return np.vstack(bank)

gunshot_bank = build_gunshot_bank(gunshot_wavs)
print("Gunshot bank shape:", gunshot_bank.shape)


Embedding gunshot dataset:   5%|▌         | 45/851 [00:00<00:03, 238.11it/s]

[gunshot] skip 3 (1).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (10).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (11).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (12).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (13).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (14).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (15).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (16).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (17).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (18).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (19).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (2).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (

Embedding gunshot dataset:  13%|█▎        | 114/851 [00:00<00:02, 308.00it/s]

[gunshot] skip 3 (59).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (6).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (60).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (61).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (62).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (63).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (64).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (65).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (66).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (67).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (68).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 (69).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 3 

Embedding gunshot dataset:  21%|██        | 178/851 [00:00<00:02, 313.36it/s]

...

Embedding gunshot dataset:  94%|█████████▎| 797/851 [00:02<00:00, 347.32it/s]

[gunshot] skip 8 (82).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (83).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (84).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (85).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (86).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (87).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (88).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (89).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (9).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (90).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (91).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 (92).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 8 

Embedding gunshot dataset: 100%|██████████| 851/851 [00:02<00:00, 315.40it/s]

[gunshot] skip 9 (77).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (78).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (79).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (8).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (80).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (81).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (82).wav: IndexError: too many indices for tensor of dimension 1
[gunshot] skip 9 (9).wav: IndexError: too many indices for tensor of dimension 1


RuntimeError: No gunshot embeddings created. Check KAGGLE_ROOT.

In [ ]:

# ========================
# 8) SCAN LAB WAVS + SAVE 5s CLIPS + SPECTROGRAMS
# ========================

def cosine_to_bank(e, bank):
    return float(np.max(bank @ e))

def format_mmss(seconds: float) -> str:
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"

def save_spectrogram(y: np.ndarray, sr: int, out_png: Path, title: str):
    y16 = resample(y, sr, 16000)
    S = librosa.stft(y16, n_fft=1024, hop_length=256)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=16000, hop_length=256, x_axis="time", y_axis="hz")
    plt.colorbar(format="%+2.0f dB")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def scan_lab_file(src_path: Path, bank: np.ndarray):
    src_path = Path(src_path)

    cached = None
    path = src_path
    if USE_LOCAL_CACHE:
        cached = ensure_local_copy(src_path, CACHE_DIR)
        if cached is None:
            return pd.DataFrame(columns=["file","time_sec","time_mmss","similarity_pct","clip_path","spec_path"])
        path = cached

    try:
        with sf.SoundFile(str(path)) as f:
            dur_sec = len(f) / f.samplerate
    except Exception as ex:
        print(f"[lab] can't open {src_path.name}: {type(ex).__name__}: {ex}")
        safe_unlink(cached)
        return pd.DataFrame(columns=["file","time_sec","time_mmss","similarity_pct","clip_path","spec_path"])

    times = np.arange(0, max(0.0, dur_sec - WINDOW_SEC) + 1e-9, HOP_SEC)

    scored = []
    for t in times:
        try:
            y, sr = load_segment(path, float(t), WINDOW_SEC)
            e = embed_panns(y, sr)
            sim = cosine_to_bank(e, bank)
            scored.append((sim, float(t), y, sr))
        except Exception:
            continue

    if not scored:
        safe_unlink(cached)
        return pd.DataFrame(columns=["file","time_sec","time_mmss","similarity_pct","clip_path","spec_path"])

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:TOP_K_PER_FILE]

    rows = []
    for sim, t, y, sr in top:
        if sim < SIM_THRESHOLD:
            continue

        pct = max(0.0, min(1.0, (sim + 1.0) / 2.0)) * 100.0
        mmss = format_mmss(t)
        stem = src_path.stem

        out_clip = CLIP_DIR / f"{stem}_t{t:.2f}_{mmss}_sim{pct:.1f}.wav"
        out_png  = SPEC_DIR / f"{stem}_t{t:.2f}_{mmss}_sim{pct:.1f}.png"

        sf.write(str(out_clip), y, sr)
        save_spectrogram(y, sr, out_png, title=f"{src_path.name} @ {mmss} (sim {pct:.1f}%)")

        rows.append({
            "file": str(src_path),
            "time_sec": float(t),
            "time_mmss": mmss,
            "similarity_pct": float(pct),
            "clip_path": str(out_clip),
            "spec_path": str(out_png),
        })

    safe_unlink(cached)
    return pd.DataFrame(rows).sort_values("similarity_pct", ascending=False)

all_rows = []
for wav in tqdm(lab_wavs, desc="Scanning lab WAVs"):
    df = scan_lab_file(wav, gunshot_bank)
    if len(df) > 0:
        all_rows.append(df)

results = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame(
    columns=["file","time_sec","time_mmss","similarity_pct","clip_path","spec_path"]
)

results = results.sort_values("similarity_pct", ascending=False)
results.head(30)


In [ ]:

# ========================
# 9) OPTIONAL: SAVE CSV
# ========================
# results.to_csv("gunshot_similarity_results.csv", index=False)
# print("Wrote gunshot_similarity_results.csv")
